In [ ]:
# Testing Cell

from aviary.api import Aircraft, Mission
from aviary.subsystems.mass.gasp_based.fuel import BWBFuselageMass, FuselageMass, TankCapacity
from aviary.utils.doctape import get_variable_name, glue_variable

glue_variable(get_variable_name(BWBFuselageMass), md_code=True)
glue_variable(get_variable_name(FuselageMass), md_code=True)
glue_variable(get_variable_name(TankCapacity), md_code=True)

glue_variable(get_variable_name(Mission.TOTAL_FUEL_MASS), md_code=True)

glue_variable(get_variable_name(Aircraft.Fuel.MAX_CAPACITY_MASS), md_code=True)

# Core Mass Subsystem

The core mass subsystem is responsible for calculating the mass properties of the aircraft.
This includes the empty weight, fuel weight, payload weight, and other mass properties.

```{note}
This section is under further development.
```

## FLOPS Based Mass Subsystem

FLOPS used a series of regressions and basic relations to estimate the weight of various transport aircraft components. The equations used are well-documented and freely available[^FLOPS_weights]. Aviary uses mass instead of weight, which is a subtle but important change to be aware of.

### Fuel_Capacity

The Fuel Capacity Group contains 4 subsystems:

1. Wing Fuel Capacity

     Aviary assumes that the wing volume can be approximated as a rectangular based pyramid, and that the available volume for fuel is a user specified fraction of this theoretcial volume (Aircraft.Fuel.WING_FUEL_FRACTION). The derivation of Aviary's theoretical volume calculation is included below:

    ![wing_fuel_capacity_derivation](./images/wing_fuel_capacity_derivation.png)

    Aviary then converts the volume to a capacity (lbm) using the user specified fuel density (Aircraft.Fuel.DENSITY).

2. Fuselage Fuel Capacity

    Fusleage Fuel Capacity  = Total Fuel Capacity - Wing Fuel Capacity

3. Auxiliary Fuel capacity

    Auxiliary Fuel Capacity = Total Fuel Capacity - Wing Fuel Capacity - Fuselage Fuel Capacity

4. Total Fuel Capacity

    Total Fuel Capacity = Wing Fuel Capacity + Fuselage Fuel Capacity + Auxiliary Fuel Capacity

It is possible for Aviary to calculate negative fuel capacities depending on user overrides and the input wing geometry. These situations are not currently flagged to the user.

# References
For additional information, please see the following resources:

[^FLOPS_weights]: The Flight Optimization System Weights Estimation Method. NASA Technology
Transfer Program. URL: https://ntrs.nasa.gov/citations/20170005851

## GASP Based Mass Subsystem

### Fuel Model
  - The GASP mass subsystem has a nonlinear solver needed to compute fuel-related variables. In order for it to converge, one must provide good initial guesses for the inputs. Otherwise, it may claim that convergence is reached but gives rise to a strange solution.
  - The computation in the {glue:md}`TankCapacity` component can not be matched in GASP Fortran code exactly. It is possible that `extra_fuel_volume` becomes negative. We added code to make sure that it is always positive.
  - **Note:** The historic name of `FUEL_MASS_REQUIRED` is `INGASP.WFAREQ`, but `WFAREQ` includes fuel margin in GASP while `FUEL_MASS_REQUIRED` doesn't. The historic name of `TOTAL_FUEL_MASS` is `INGASP.WFA`, but does not include fuel margin in GASP.
  - **Note:** GASP Fortran code has features that are not implemented in Aviary (e.g. tail boom support, tip tank weight, fuselage acoustic treatment, pylon, acoustic treatment).

The fuel computation is a nonlinear system of equations. A simplied XDSM diagram is shown below:

![GASP based fuel mass computation](./images/gasp_fuel_mass.png)

A Newton solver is applied until `FUEL_MASS`, `wingfuel_mass_min` and {glue:md}`Aircraft.Fuel.MAX_CAPACITY_MASS` are converged.

For conventional aircraft, {glue:md}`BWBFuselageMass` is replaced by {glue:md}`FuselageMass`.